# 6. Ekstraksi Fitur CO & SO2 (TSFEL)

Setelah data berhasil dikumpulkan dan divisualisasikan pada tahapan sebelumnya,
langkah berikutnya adalah melakukan **ekstraksi fitur** terhadap sinyal deret waktu
polutan CO dan SO2. Ekstraksi fitur dilakukan menggunakan pustaka
[TSFEL](https://tsfel.readthedocs.io/) (Time Series Feature Extraction Library)
untuk mengekstraksi karakteristik sinyal pada domain **statistical**, **temporal**, dan **spectral**.

Data yang digunakan merupakan data harian hasil pengumpulan dari Sentinel-5P melalui openEO,
periode **September 2025 s.d. Agustus 2026** di wilayah
**Desa Bundah, Kecamatan Sreseh, Kabupaten Sampang** (total 362 baris data).

```{note}
Library yang digunakan:
`pip install tsfel`
```

## 6.1 Ekstraksi Fitur CO

Tahapan analisis fitur CO dimulai dari pembacaan data, pembersihan nilai outlier,
imputasi data hilang (missing value), hingga pemrosesan ekstraksi fitur dengan TSFEL.

Pada tahap awal, kolom `co` dipastikan bertipe numerik dengan `pd.to_numeric(..., errors='coerce')`
agar nilai teks/error dikonversi menjadi `NaN`. Data juga diurutkan berdasarkan tanggal
untuk memastikan urutan kronologis yang konsisten.

In [1]:
import pandas as pd
import numpy as np
import inspect
import tsfel.feature_extraction.features as tsfel_features

# ---------- 1. Muat dan bersihkan data ----------
# Membaca data polutan CO
df_co = pd.read_csv('../data/kualitas_udara_bundah_sreseh.csv', parse_dates=['date'])
print('Kolom dataset:', df_co.columns.tolist())
df_co = df_co.sort_values('date').reset_index(drop=True)

target_co = 'co'
# Memastikan kolom target bertipe numerik, error menjadi NaN
df_co[target_co] = pd.to_numeric(df_co[target_co], errors='coerce')
n_missing_before_co = df_co[target_co].isna().sum()
print(f"Jumlah nilai non-numerik/kosong awal yang dikonversi jadi NaN (CO): {n_missing_before_co}")

Index(['date', 'no2', 'co'], dtype='object')
Jumlah nilai non-numerik/kosong awal yang dikonversi jadi NaN (CO): 190


### Tahap 2: Deteksi Outliers dengan Metode IQR - CO

Deteksi nilai pencilan (outlier) dilakukan menggunakan pendekatan statistik **Interquartile Range (IQR)**.
Nilai dianggap outlier jika berada di luar batas $[Q1 - 1{,}5 \times IQR,\ Q3 + 1{,}5 \times IQR]$.

Berdasarkan distribusi data CO:
- **Batas Bawah IQR**: 0,0180
- **Batas Atas IQR**: 0,0389
- **Jumlah Outlier**: 1 data terdeteksi melampaui batas

Nilai outlier tersebut diubah menjadi `NaN` agar nantinya dapat diisi kembali secara proporsional
bersama data kosong lainnya pada tahap imputasi.

In [2]:
# ---------- 2. Deteksi dan Penghapusan Outliers (Pencilan) - CO ----------
# Menghitung Kuartil 1 (Q1) dan Kuartil 3 (Q3)
Q1_co = df_co[target_co].quantile(0.25)
Q3_co = df_co[target_co].quantile(0.75)
# Menghitung Interquartile Range (IQR)
IQR_co = Q3_co - Q1_co
# Menentukan batas kewajaran data
lower_bound_co = Q1_co - 1.5 * IQR_co
upper_bound_co = Q3_co + 1.5 * IQR_co
print(f"Batas Bawah IQR (CO): {lower_bound_co:.4f} | Batas Atas IQR (CO): {upper_bound_co:.4f}")
# Menghapus nilai yang melanggar batas (diubah menjadi NaN)
outliers_co = (df_co[target_co] < lower_bound_co) | (df_co[target_co] > upper_bound_co)
df_co.loc[outliers_co, target_co] = np.nan
print(f"Jumlah outliers yang terdeteksi dan dikosongkan (CO): {outliers_co.sum()}")

Batas Bawah IQR (CO): 0.0180 | Batas Atas IQR (CO): 0.0389
Jumlah outliers yang terdeteksi dan dikosongkan (CO): 1


### Tahap 3: Imputasi Missing Value - CO

Data rekaman satelit Sentinel-5P sering memiliki kekosongan nilai akibat tutupan awan.
Sebelum imputasi, terdapat total **191 nilai kosong** (190 missing value awal + 1 outlier yang dikosongkan).

Imputasi dilakukan menggunakan **interpolasi berbasis waktu** (`interpolate(method='time')`)
yang menghubungkan titik sebelum dan sesudah data kosong secara linear berdasarkan jarak tanggal.
Selanjutnya diterapkan `ffill()` dan `bfill()` untuk memastikan tidak ada celah di ujung deret.

Hasil akhirnya: **sisa missing value = 0** (data 100% terisi penuh, memenuhi syarat batas maksimal missing value).

In [3]:
# ---------- 3. Imputasi Missing Value - CO ----------
# Jadikan kolom tanggal sebagai index sementara untuk interpolasi
df_co_clean = df_co.set_index('date')
# Melakukan interpolasi berbasis waktu
df_co_clean = df_co_clean.interpolate(method='time')
# Menambal celah di awal atau akhir data jika interpolasi tidak menjangkau
df_co_clean = df_co_clean.ffill().bfill()
print(f"Sisa missing value setelah proses imputasi (CO): {df_co_clean[target_co].isna().sum()}")

Sisa missing value setelah proses imputasi (CO): 0


### Tahap 4: Persiapan Data untuk TSFEL - CO

Data CO yang sudah bersih diubah ke bentuk array NumPy 1 dimensi (`signal_co`).
Parameter frekuensi sampling ditentukan `fs = 1` (1 data per hari).

Disiapkan daftar **68 fitur** TSFEL yang mencakup:
- **Domain Statistical** (20 fitur): mean, variance, standard deviation, skewness, kurtosis, entropy, dsb.
- **Domain Temporal** (15 fitur): autocorrelation, slope, zero crossing rate, mean absolute difference, dsb.
- **Domain Spectral** (26 fitur): spectral entropy, spectral centroid, MFCC, wavelet energy/entropy, dsb.
- **Domain Fractal** (6 fitur) & **Domain Lainnya** (1 fitur).

In [4]:
fs = 1
signal_co = df_co_clean[target_co].astype(float).values

# Daftar PERSIS fitur (68 Fitur)
FEATURE_LIST = """abs_energy auc autocorr average_power calc_centroid calc_max calc_mean calc_median calc_min calc_std calc_var dfa distance ecdf ecdf_percentile ecdf_percentile_count ecdf_slope entropy fundamental_frequency higuchi_fractal_dimension hist_mode human_range_energy hurst_exponent interq_range kurtosis lempel_ziv lpcc max_frequency max_power_spectrum maximum_fractal_length mean_abs_deviation mean_abs_diff mean_diff median_abs_deviation median_abs_diff median_diff median_frequency mfcc mse negative_turning neighbourhood_peaks petrosian_fractal_dimension pk_pk_distance positive_turning power_bandwidth rms skewness slope spectral_centroid spectral_decrease spectral_distance spectral_entropy spectral_kurtosis spectral_positive_turning spectral_roll_off spectral_roll_on spectral_skewness spectral_slope spectral_spread spectral_variation spectrogram_mean_coeff sum_abs_diff wavelet_abs_mean wavelet_energy wavelet_entropy wavelet_std wavelet_var zero_cross""".split()

print("Jumlah fitur yang disiapkan untuk diekstrak:", len(FEATURE_LIST))

Jumlah fitur yang disiapkan untuk diekstrak: 68


### Tahap 5: Eksekusi TSFEL dan Penyimpanan Hasil - CO

Ekstraksi ke-68 fitur dijalankan secara dinamis dengan bantuan fungsi:
- `to_scalar`: memastikan setiap hasil fitur berupa nilai skalar numerik tunggal (mengambil rata-rata jika output berbentuk array).
- `extract_one`: memanggil fungsi dari modul `tsfel.feature_extraction.features` sesuai parameter yang dibutuhkan (`fs` atau hanya sinyal).

Seluruh hasil dirangkum menjadi DataFrame berukuran 1 baris x 68 kolom dan disimpan ke file `fitur_co_bundah_sreseh.csv`.

In [5]:
# ---------- 5. Proses Ekstraksi Fitur - CO ----------
def to_scalar(result):
    # Mengubah hasil dictionary dari TSFEL menjadi nilai mentah
    if isinstance(result, dict) and "values" in result:
        result = result["values"]
    # Jika hasil berupa array atau list panjang, kita ambil rata-ratanya
    if isinstance(result, (list, tuple, np.ndarray)):
        arr = np.asarray(result, dtype=float)
        return float(np.nanmean(arr))
    # Jika sudah skalar, ubah jadi float standar
    return float(result)

def extract_one(fn_name, signal, fs):
    # Memanggil fungsi TSFEL secara dinamis berdasarkan nama fiturnya
    fn = getattr(tsfel_features, fn_name)
    params = inspect.signature(fn).parameters
    # Memeriksa apakah fungsi tersebut butuh parameter 'fs'
    if "fs" in params:
        result = fn(signal, fs)
    else:
        result = fn(signal)
    return to_scalar(result)

# Menjalankan iterasi untuk setiap fitur pada sinyal CO
row_co = {}
for fn_name in FEATURE_LIST:
    row_co[fn_name] = extract_one(fn_name, signal_co, fs)

extracted_co = pd.DataFrame([row_co])
print(f"Berhasil! Ekstraksi CO menghasilkan {extracted_co.shape[1]} fitur.")

# Menyimpan hasil ke dalam format CSV
output_co = '../data/fitur_co_bundah_sreseh.csv'
extracted_co.to_csv(output_co, index=False)
print(f"File berhasil disimpan sebagai: {output_co}")

Berhasil! Ekstraksi CO menghasilkan 68 fitur.
File berhasil disimpan sebagai: ../data/fitur_co_bundah_sreseh.csv


---

## 6.2 Ekstraksi Fitur SO2

Tahapan pada polutan SO2 mengikuti metodologi dan urutan yang sama dengan CO.
Data SO2 diperoleh melalui crawling dari Sentinel-5P via openEO pada sel di bawah ini,
kemudian digabungkan ke file dataset utama `kualitas_udara_bundah_sreseh.csv`.

Setelah kolom SO2 tersedia, proses dilanjutkan ke deteksi outlier, imputasi, dan ekstraksi 68 fitur TSFEL.

### Pengambilan Data SO2 dari openEO (Sentinel-5P)

Pengambilan data SO2 menggunakan koleksi **SENTINEL_5P_L2** band `SO2` melalui koneksi openEO
Copernicus Data Space Ecosystem (CDSE) dengan batasan spasial poligon Desa Bundah
dan rentang waktu September 2025 s.d. Agustus 2026.

In [7]:
# Pengambilan data SO2 via openEO CDSE
import openeo
import json

with open('../geojson/bundah_sreseh_sampang.geojson') as f:
    aoi = json.load(f)

START_DATE = '2025-09-01'
END_DATE   = '2026-08-31'

connection = openeo.connect('openeofed.dataspace.copernicus.eu')
connection.authenticate_oidc()

cube_so2 = connection.load_collection(
    'SENTINEL_5P_L2',
    spatial_extent=aoi,
    temporal_extent=[START_DATE, END_DATE],
    bands=['SO2'],
)
so2_ts = cube_so2.aggregate_spatial(geometries=aoi, reducer='mean')

so2_job = so2_ts.create_job(title='so2_bundah_sreseh_1thn', out_format='JSON')
so2_job.start_and_wait()
so2_job.get_results().download_file('../data/so2_raw.json')
print('so2_raw.json berhasil diunduh!')

Authenticated using refresh token.
0:00:00 Job 'cdse-j-2609171611334dd4b36f82385d0ebc0d': send 'start'
0:00:06 Job 'cdse-j-2609171611334dd4b36f82385d0ebc0d': queued (progress 0%)
0:00:11 Job 'cdse-j-2609171611334dd4b36f82385d0ebc0d': queued (progress 0%)
0:00:18 Job 'cdse-j-2609171611334dd4b36f82385d0ebc0d': queued (progress 0%)
0:00:26 Job 'cdse-j-2609171611334dd4b36f82385d0ebc0d': queued (progress 0%)
0:00:37 Job 'cdse-j-2609171611334dd4b36f82385d0ebc0d': queued (progress 0%)
0:00:49 Job 'cdse-j-2609171611334dd4b36f82385d0ebc0d': running (progress N/A)
0:01:05 Job 'cdse-j-2609171611334dd4b36f82385d0ebc0d': running (progress N/A)
0:01:24 Job 'cdse-j-2609171611334dd4b36f82385d0ebc0d': running (progress N/A)
0:01:49 Job 'cdse-j-2609171611334dd4b36f82385d0ebc0d': running (progress N/A)
0:02:19 Job 'cdse-j-2609171611334dd4b36f82385d0ebc0d': running (progress N/A)
0:02:57 Job 'cdse-j-2609171611334dd4b36f82385d0ebc0d': running (progress N/A)
0:03:44 Job 'cdse-j-2609171611334dd4b36f82385d0eb

### Parsing dan Penggabungan Data SO2 ke Dataset

Hasil unduhan `so2_raw.json` di-parse menjadi deret waktu bertanggal,
lalu digabungkan dengan file `kualitas_udara_bundah_sreseh.csv` berdasarkan kolom `date`.
Dengan demikian dataset memiliki kolom lengkap: `date`, `no2`, `co`, dan `so2`.

In [8]:
import json
import pandas as pd

# Parse so2_raw.json
def load_openeo_timeseries(path, colname):
    with open(path) as f:
        raw = json.load(f)
    rows = []
    for ts, values in raw.items():
        v = values[0][0] if values and values[0] else None
        rows.append({'date': ts[:10], colname: v})
    return pd.DataFrame(rows)

df_so2_raw = load_openeo_timeseries('../data/so2_raw.json', 'so2')
df_so2_raw['date'] = pd.to_datetime(df_so2_raw['date'])
df_so2_raw = df_so2_raw.sort_values('date').reset_index(drop=True)
print('Data SO2 dari openEO:', df_so2_raw.shape)
print(df_so2_raw.head())

# Gabungkan ke CSV utama
df_main = pd.read_csv('../data/kualitas_udara_bundah_sreseh.csv', parse_dates=['date'])
if 'so2' not in df_main.columns:
    df_main = df_main.merge(df_so2_raw[['date', 'so2']], on='date', how='left')
    df_main.to_csv('../data/kualitas_udara_bundah_sreseh.csv', index=False)
    print('Kolom so2 berhasil ditambahkan ke CSV!')
else:
    print('Kolom so2 sudah ada di CSV, tidak perlu diperbarui.')
print('Kolom CSV sekarang:', df_main.columns.tolist())

Data SO2 dari openEO: (350, 2)
        date       so2
0 2025-09-01  0.000200
1 2025-09-02       NaN
2 2025-09-03 -0.000045
3 2025-09-04 -0.000011
4 2025-09-05 -0.000127
Kolom so2 berhasil ditambahkan ke CSV!
Kolom CSV sekarang: ['date', 'no2', 'co', 'so2']


In [9]:
# ---------- 1. Muat dan bersihkan data - SO2 ----------
# Membaca data polutan SO2
df_so2 = pd.read_csv('../data/kualitas_udara_bundah_sreseh.csv', parse_dates=['date'])
print('Kolom dataset:', df_so2.columns.tolist())
df_so2 = df_so2.sort_values('date').reset_index(drop=True)

target_so2 = 'so2'
# Memastikan kolom target bertipe numerik, error menjadi NaN
df_so2[target_so2] = pd.to_numeric(df_so2[target_so2], errors='coerce')
n_missing_before_so2 = df_so2[target_so2].isna().sum()
print(f"Jumlah nilai non-numerik/kosong awal yang dikonversi jadi NaN (SO2): {n_missing_before_so2}")

Index(['date', 'no2', 'co', 'so2'], dtype='object')
Jumlah nilai non-numerik/kosong awal yang dikonversi jadi NaN (SO2): 124


### Tahap 2: Deteksi Outliers dengan Metode IQR - SO2

Metode IQR diterapkan pada kolom `so2` untuk menemukan lonjakan nilai abnormal:
- **Batas Bawah IQR**: -0,0003
- **Batas Atas IQR**: 0,0003
- **Jumlah Outlier**: 11 data terdeteksi dan dikosongkan (diubah menjadi `NaN`)

Rentang yang rapat ini mengindikasikan bahwa sebagian besar nilai konsentrasi SO2 berkumpul di sekitar nol,
sehingga anomali akibat gangguan sensor atau tutupan awan dapat teridentifikasi dengan baik.

In [10]:
# ---------- 2. Deteksi dan Penghapusan Outliers (Pencilan) - SO2 ----------
# Menghitung Kuartil 1 (Q1) dan Kuartil 3 (Q3)
Q1_so2 = df_so2[target_so2].quantile(0.25)
Q3_so2 = df_so2[target_so2].quantile(0.75)
# Menghitung Interquartile Range (IQR)
IQR_so2 = Q3_so2 - Q1_so2
# Menentukan batas kewajaran data
lower_bound_so2 = Q1_so2 - 1.5 * IQR_so2
upper_bound_so2 = Q3_so2 + 1.5 * IQR_so2
print(f"Batas Bawah IQR (SO2): {lower_bound_so2:.4f} | Batas Atas IQR (SO2): {upper_bound_so2:.4f}")
# Menghapus nilai yang melanggar batas (diubah menjadi NaN)
outliers_so2 = (df_so2[target_so2] < lower_bound_so2) | (df_so2[target_so2] > upper_bound_so2)
df_so2.loc[outliers_so2, target_so2] = np.nan
print(f"Jumlah outliers yang terdeteksi dan dikosongkan (SO2): {outliers_so2.sum()}")

Batas Bawah IQR (SO2): -0.0003 | Batas Atas IQR (SO2): 0.0003
Jumlah outliers yang terdeteksi dan dikosongkan (SO2): 11


### Tahap 3: Imputasi Missing Value - SO2

Pada data SO2, terdapat **124 nilai kosong awal** dan **11 outlier**, menghasilkan total **135 NaN** yang harus diisi.

Menggunakan metode interpolasi waktu (`interpolate(method='time')`) beserta `ffill()` dan `bfill()`,
seluruh data berhasil diisi sehingga **sisa missing value = 0**.
Data kini siap diekstraksi tanpa kendala celah kosong.

In [11]:
# ---------- 3. Imputasi Missing Value - SO2 ----------
# Jadikan kolom tanggal sebagai index sementara untuk interpolasi
df_so2_clean = df_so2.set_index('date')
# Melakukan interpolasi berbasis waktu
df_so2_clean = df_so2_clean.interpolate(method='time')
# Menambal celah di awal atau akhir data jika interpolasi tidak menjangkau
df_so2_clean = df_so2_clean.ffill().bfill()
print(f"Sisa missing value setelah proses imputasi (SO2): {df_so2_clean[target_so2].isna().sum()}")

Sisa missing value setelah proses imputasi (SO2): 0


### Tahap 4: Persiapan Data untuk TSFEL - SO2

Sinyal SO2 bersih dikonversi ke array 1 dimensi (`signal_so2`) dengan parameter $fs = 1$.
Daftar 68 fitur (`FEATURE_LIST`) yang digunakan sama persis dengan ekstraksi CO sebelumnya.

In [12]:
signal_so2 = df_so2_clean[target_so2].astype(float).values
print("Jumlah fitur yang disiapkan untuk diekstrak:", len(FEATURE_LIST))

Jumlah fitur yang disiapkan untuk diekstrak: 68


### Tahap 5: Eksekusi TSFEL dan Penyimpanan Hasil - SO2

Ekstraksi fitur dijalankan menggunakan fungsi `to_scalar` dan `extract_one` pada sinyal SO2.
Hasil akhir berupa 1 baris x 68 fitur numerik disimpan ke file `fitur_so2_bundah_sreseh.csv`.

In [13]:
# ---------- 5. Proses Ekstraksi Fitur - SO2 ----------
# Menjalankan iterasi untuk setiap fitur pada sinyal SO2
row_so2 = {}
for fn_name in FEATURE_LIST:
    row_so2[fn_name] = extract_one(fn_name, signal_so2, fs)

extracted_so2 = pd.DataFrame([row_so2])
print(f"Berhasil! Ekstraksi SO2 menghasilkan {extracted_so2.shape[1]} fitur.")

# Menyimpan hasil ke dalam format CSV
output_so2 = '../data/fitur_so2_bundah_sreseh.csv'
extracted_so2.to_csv(output_so2, index=False)
print(f"File berhasil disimpan sebagai: {output_so2}")

Berhasil! Ekstraksi SO2 menghasilkan 68 fitur.
File berhasil disimpan sebagai: ../data/fitur_so2_bundah_sreseh.csv


## 6.3 Pengelompokan Fitur Berdasarkan Domain

Berikut adalah rekapitulasi ke-68 fitur yang berhasil diekstrak berdasarkan domain resminya di TSFEL:

| Domain | Jumlah Fitur | Deskripsi Singkat |
|---|:---:|---|
| **Spectral** | 26 | Karakteristik frekuensi sinyal (FFT, spectral centroid, spectral entropy, wavelet) |
| **Statistical** | 20 | Ukuran pemusatan dan penyebaran data (mean, median, std, variance, skewness, kurtosis) |
| **Temporal** | 15 | Pola perubahan terhadap waktu (autocorrelation, slope, zero crossing rate, diff) |
| **Fractal** | 6 | Kompleksitas dan dimensi fraktal deret waktu (Higuchi, Petrosian, DFA, Hurst) |
| **Unknown / Lainnya** | 1 | Fitur turunan ECDF slope |
| **Total** | **68** | Fitur lengkap siap untuk pemodelan machine learning |


In [14]:
import tsfel

# Ambil metadata domain resmi dari TSFEL (statistical/temporal/spectral),
# lalu cocokkan ke nama fungsi snake_case yang kita pakai di FEATURE_LIST
cfg = tsfel.get_features_by_domain()

func_to_domain = {}
for domain, feats in cfg.items():
    for feat_key, meta in feats.items():
        func_path = meta.get("function", "")
        func_name = func_path.split(".")[-1]
        func_to_domain[func_name] = domain

feature_domain_map = pd.Series(
    {fn: func_to_domain.get(fn, "unknown") for fn in FEATURE_LIST}, name="domain"
)
feature_domain_map.value_counts()

domain
spectral       26
statistical    20
temporal       15
fractal         6
unknown         1
Name: count, dtype: int64

In [15]:
# Daftar lengkap fitur per domain (untuk dokumentasi laporan)
for domain in ["statistical", "temporal", "spectral", "unknown"]:
    kolom = feature_domain_map[feature_domain_map == domain].index.tolist()
    print(f"\n=== {domain.upper()} ({len(kolom)} fitur) ===")
    print(kolom)


=== STATISTICAL (20 fitur) ===
['abs_energy', 'average_power', 'calc_max', 'calc_mean', 'calc_median', 'calc_min', 'calc_std', 'calc_var', 'ecdf', 'ecdf_percentile', 'ecdf_percentile_count', 'entropy', 'hist_mode', 'interq_range', 'kurtosis', 'mean_abs_deviation', 'median_abs_deviation', 'pk_pk_distance', 'rms', 'skewness']

=== TEMPORAL (15 fitur) ===
['auc', 'autocorr', 'calc_centroid', 'distance', 'lempel_ziv', 'mean_abs_diff', 'mean_diff', 'median_abs_diff', 'median_diff', 'negative_turning', 'neighbourhood_peaks', 'positive_turning', 'slope', 'sum_abs_diff', 'zero_cross']

=== SPECTRAL (26 fitur) ===
['fundamental_frequency', 'human_range_energy', 'lpcc', 'max_frequency', 'max_power_spectrum', 'median_frequency', 'mfcc', 'power_bandwidth', 'spectral_centroid', 'spectral_decrease', 'spectral_distance', 'spectral_entropy', 'spectral_kurtosis', 'spectral_positive_turning', 'spectral_roll_off', 'spectral_roll_on', 'spectral_skewness', 'spectral_slope', 'spectral_spread', 'spectral_va

## 6.4 Kesimpulan

1. **Pembersihan Data & Imputasi**:
   - Sinyal CO memiliki 1 outlier (batas normal [0,0180; 0,0389]) dan total 191 data kosong sebelum imputasi.
   - Sinyal SO2 memiliki 11 outlier (batas normal [-0,0003; 0,0003]) dan total 135 data kosong sebelum imputasi.
   - Metode interpolasi waktu berhasil mengisi seluruh data kosong sehingga **sisa missing value = 0** (0%), memenuhi batas toleransi kualitas data.

2. **Ekstraksi Fitur TSFEL**:
   - Sebanyak **68 fitur numerik** berhasil diekstraksi untuk masing-masing polutan (CO dan SO2).
   - Fitur terdistribusi ke domain spectral (26), statistical (20), temporal (15), fractal (6), dan turunan lainnya (1).

3. **Penyimpanan Hasil**:
   - Fitur CO tersimpan di `../data/fitur_co_bundah_sreseh.csv`.
   - Fitur SO2 tersimpan di `../data/fitur_so2_bundah_sreseh.csv`.
   - Dataset terintegrasi tersimpan di `../data/kualitas_udara_bundah_sreseh.csv` lengkap dengan kolom NO2, CO, dan SO2.